In [ ]:
%load_ext autoreload

In [ ]:
%autoreload 2
import torch
import torch.nn as nn
import torch.utils.data as Data
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

## Our fancy new modules
from subgrid_parameterization.arch import Clipped_ANN
from subgrid_parameterization.preprocess.torch_helpers import split_dataset
import subgrid_parameterization.util.plot_helpers as plot_helpers
from subgrid_parameterization.preprocess.preprocess_C14 import preprocess
from subgrid_parameterization.train import Trainer

import numpy as np
from sklearn.metrics import r2_score

In [ ]:
path0 = "../test/"

files = ["data/BOMEX_pruned_thinned"]

save_name = "C14_BOMEXtest"

input, output, aux_dict = preprocess(files, data_root=path0)

aux_dict["files"] = files

In [ ]:
dataset = Data.TensorDataset(torch.tensor(input), torch.tensor(output))
train_dataset, test_dataset = split_dataset(
    dataset, 0.8
)  # random_split() returns a Subset so doesn't work later

In [ ]:
_, C14train = train_dataset.tensors
# np.array([numerator/(np.std(np.array(output)[:,i])) for i in range(np.array(output).shape[1])])
lossweights = np.ones(C14train.detach().numpy().shape[1])

In [ ]:
config = {
    "batch_size": 48,
    "lr": 0.001,  ## learning rate
    "wd": 0.01,  ## weight decay
    "epochs": 2000,  ## Setting this to a high number because early stopping
    # "subsample":10,   ## Take a subsample of 1000 data points
    "patience": 20,  ## Patience for early stopping
}

beta1 = 0.5
beta2 = 0.999

In [ ]:
# use GPUs if available
if torch.cuda.is_available():
    print("CUDA Available")
    device = torch.device("cuda")
else:
    print("CUDA Not Available")
    device = torch.device("cpu")

In [ ]:
## Construct training and validation dataloaders
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
valid_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=True)

In [ ]:
nvars = 5  # up2,vp2,wp2,Lup,Ldown
iso = True
if iso:
    nvarsout = 1
else:
    nvarsout = 2

N = [nvars, 8, 4, nvarsout]

try:
    del model
    model = Clipped_ANN(
        N=N,
        clamping_range=[0, 2],
    ).double()  ## NN architecture: could be ANN w/o clipping
except:
    model = Clipped_ANN(
        N=N, clamping_range=[0, 2]
    ).double()  ## NN architecture: could be ANN w/o clipping

model.to(device)
config["learnable parameters"] = sum(p.numel() for p in model.parameters())

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
)  # weight_decay=config["wd"], betas=(beta1, beta2))
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.3, patience=10)
criterion = nn.MSELoss()

In [ ]:
trainer = Trainer(config=config, device=device, lossweights=lossweights)

best_model = trainer.train_loop(
    model, optimizer, train_loader, valid_loader, save_name=save_name
)

In [ ]:
cutStartPct = 0.5
zoomEndEpochs = 2 * config["patience"]
plot_helpers.plot_losses(
    [trainer.test_loss, trainer.train_loss],
    ["Test", "Train"],
    cutStartPct,
    zoomEndEpochs,
)

In [ ]:
x_test, y_test = test_dataset.tensors
# x_test=x_test.to(device)
y_test = y_test.squeeze().detach().cpu().numpy()
y_pred = best_model(x_test).squeeze().detach().cpu().numpy()

In [ ]:
y_text = r"$C_{14}$"

val_r2 = r2_score(y_test, y_pred)
val_r = np.corrcoef(y_test, y_pred)[0, 1]

print(r"Statistics for " + y_text)
print("R^2: %.4f" % np.mean(val_r2))
print("Correlation: %.4f" % +np.mean(val_r) + "\n")

In [ ]:
l1 = "Predicted"
l2 = "True"
nvar = 1
# z=zt[0,:]
y_text = r"$C_{14}$"

# fig1,ax1 = plt.subplots(1,nvar,figsize = (12, 6))
fig2, ax2 = plt.subplots(1, nvar, figsize=(12, 6))
# fig3,ax3 = plt.subplots(1,nvar,figsize = (12, 6))

ax2.scatter(y_test, y_pred)
xmin, xmax = ax2.get_xlim()
ymin, ymax = ax2.get_ylim()
ax2.plot([xmin, xmax], [xmin, xmax])
ax2.set_xlim([xmin, xmax])
# ax2.set_xlim([0,2])
# ax2.set_ylim([0,2])
ax2.set_xlabel(l2)
ax2.set_ylabel(l1)
ax2.set_title(y_text, fontsize=20)

In [ ]:
x_test, y_test = train_dataset.tensors
# x_train   =x_train.to(device)
y_test = y_test.squeeze().detach().cpu().numpy()
y_pred = best_model(x_test).squeeze().detach().cpu().numpy()

y_text = r"$C_{14}$"
train_r2 = r2_score(y_test, y_pred)
train_r = np.corrcoef(y_test, y_pred)[0, 1]

print("Statistics for " + y_text)
print("R^2: %.4f" % np.mean(train_r2))
print("Correlation: %.4f" % +np.mean(train_r) + "\n")

fig2, ax2 = plt.subplots(1, nvar, figsize=(12, 6))
# fig3,ax3 = plt.subplots(1,nvar,figsize = (12, 6))

ax2.scatter(y_test, y_pred)
xmin, xmax = ax2.get_xlim()
ymin, ymax = ax2.get_ylim()
ax2.plot([xmin, xmax], [xmin, xmax])
ax2.set_xlim([xmin, xmax])
# ax2.set_xlim([0,2])
# ax2.set_ylim([0,2])
ax2.set_xlabel(l2)
ax2.set_ylabel(l1)
ax2.set_title(y_text, fontsize=20)

In [ ]:
# y_text=[r"$\overline{u'w'}/u_*^2$",r"$\overline{v'w'}/u_*^2$"]
# y_text=["U2DFSN","V2DFSN"]
# y_text[r'C_{14}']
# fig1,fig2,fig3=quickPlots(best_model,test_dataset,y_text,device)

## Example for saving the model with metadata

In [ ]:
# Model to be saved and example input tensor
from subgrid_parameterization.train.save import save_model


model = best_model  # This should be the trained model you want to save

input_vars = [
    {"name": "up2", "desc": "Variance of the x wind component"},
    {"name": "vp2", "desc": "Variance of the y wind component"},
    {"name": "wp2", "desc": "Variance of the z wind component"},
    {"name": "Lup", "desc": "Upward mixing length scale"},
    {"name": "Ldown", "desc": "Downward mixing length scale"}
]
output_vars = [{"name": "C_14", "desc": "Predicted C_14 coefficient"}]

metrics = {
    "loss": float(trainer.test_loss[-1]),
    "train_R2": train_r2,
    "val_R2": val_r2,
    "steps": len(trainer.test_loss),
    "early_stop_time": 325
}

train_config = {
    "train_dataset": "/data/train_dataset.nc",
    # "val_dataset": "/data/val_dataset.nc", split of validation set from train dataset
    "Hscale": 1000.0,
}
other_notes = "Training completed successfully with early stopping"

save_model(
    model=model,
    save_dir="./saved_models",
    filename="C14_BOMEX_example",
    input_example=None,  # You can provide an example input tensor if needed
    input_vars=input_vars,
    output_vars=output_vars,
    metrics=metrics,
    train_config=train_config,
    other_notes=other_notes
)